# PRL GENIE per-mode systematic summary

Loads outputs from the PRL GENIE per-knob pipeline (`run_prl_genie_per_knob.sh` / `run_prl_genie_chain_from_mec.sh`):

- **Per-mode merge:** `merged/<MODE>/genie_syst_<MODE>.npz` (`syst[knob][var][rate|xsec][cov|cov_frac|corr]`)
- **Final aggregate (optional):** `syst_disk/GENIE/cov_mat_dict.pkl` and `per_knob/`

For each interaction **mode** and each **knob**:

1. **Matrix figures** — fractional covariance and correlation for **rate** and **xsec** (all PRL variables).
2. **Uncertainty breakdown** — per-knob curves plus mode total (sqrt(diag of summed fractional cov)), matching the style of `systematics-summary.ipynb`.

Set `WORK_DIR` to the PRL work tree on `/exp/sbnd/data`.

In [ ]:
import os
import pickle
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

import sys
sys.path.insert(0, '/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')

from analysis_village.numucc_1p0pi.dataset_locations import GENIE_GROUP_KNOBS, GENIE_GROUP_ORDER, PLOTS_BASE
from analysis_village.numucc_1p0pi.final_selected_evt_vars import CORE_SELECTED_EVT_VARIABLE_CONFIGS
from analysis_village.numucc_1p0pi.scripts.prl_genie_per_knob import PRL_VAR_SAVE_NAMES
from analysis_village.numucc_1p0pi.utils import plot_heatmap
from analysis_village.numucc_1p0pi import utils as _numu_utils

In [ ]:
# ----------------------------
# User configuration
# ----------------------------

WORK_DIR = Path(os.environ.get(
    'PRL_GENIE_WORK_DIR',
    '/exp/sbnd/data/users/munjung/PRL_data/PRL_genie_per_knob_Ar23p',
))
OUT_DIR = Path(PLOTS_BASE) / 'prl_genie_syst_summary'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# None = all PRL variables; or e.g. ['integrated', 'muon-p']
VARS_TO_PLOT = None

# None = every mode with a merged NPZ (or pickle entry); or e.g. ['CCQE', 'MEC']
MODES_TO_PLOT = None

SAVE_FIGS = True
SHOW_FIGS = False

# Matrix heatmaps: one PDF per knob (all variables as pages). Set False to skip.
PLOT_KNOB_MATRIX_PDFS = True
# Skip scalar integrated (1×1) matrix pages; still in breakdown plots.
PLOT_INTEGRATED_MATRICES = False
SAVE_MATRIX_PDF = True
SAVE_SUMMARY_PDF = False  # summary plots: PNG only (much faster)

BREAKDOWN_FIGSIZE = (8.0, 6.0)
BREAKDOWN_FIG_DPI = 100
SUMMARY_SAVE_DPI = int(_numu_utils.dpi)
MATRIX_SAVE_DPI = 100
BREAKDOWN_SUBPLOT = dict(left=0.11, right=0.99, top=0.94, bottom=0.40)
BREAKDOWN_SUBPLOT_ONPLOT = dict(left=0.11, right=0.99, top=0.94, bottom=0.11)
LEGEND_OUTSIDE_THRESHOLD = 20

plt.rcParams['figure.figsize'] = BREAKDOWN_FIGSIZE
plt.rcParams['figure.dpi'] = BREAKDOWN_FIG_DPI

vc_by_name = {vc.var_save_name: vc for vc in CORE_SELECTED_EVT_VARIABLE_CONFIGS}
var_configs = [vc_by_name[n] for n in PRL_VAR_SAVE_NAMES if n in vc_by_name]
if VARS_TO_PLOT:
    var_configs = [vc_by_name[n] for n in VARS_TO_PLOT if n in vc_by_name]

print('WORK_DIR =', WORK_DIR)
print('OUT_DIR  =', OUT_DIR)
print('variables:', [vc.var_save_name for vc in var_configs])

In [ ]:
# ----------------------------
# Loaders + plotting helpers (PRL merged-NPZ layout)
# ----------------------------

def _sum_cov_frac_matrices(parts):
    parts = [np.asarray(p, dtype=np.float64) for p in parts if p is not None]
    if not parts:
        return None
    out = np.zeros_like(parts[0])
    for p in parts:
        out += p
    return out


def frac_unc_pct(cov_frac):
    c = np.asarray(cov_frac, dtype=np.float64)
    d = np.maximum(np.diag(c), 0.0)
    return 100.0 * np.sqrt(d)


def frac_weights_for_plot(cov_frac, var_config):
    w = frac_unc_pct(cov_frac)
    if getattr(var_config, 'var_save_name', None) == 'integrated' and len(w):
        w = np.full_like(w, float(w[0]))
    return w


def integrated_rate_frac_variance(cov_frac):
    c = np.asarray(cov_frac, dtype=np.float64)
    ones = np.ones(c.shape[0], dtype=np.float64)
    return float(ones @ c @ ones)


_GENIE_KNOB_STRIP_PREFIXES = (
    'GENIEReWeight_SBNNuSyst_multisigma_EDepFSI_',
    'GENIEReWeight_SBNNuSyst_multisigma_',
    'GENIEReWeight_SBN_v1_multisim_',
    'GENIEReWeight_SBN_v1_',
    'GENIEReWeight_',
    'CCQETemplateReweight_',
)
_GENIE_KNOB_STRIP_TOKENS = ('_multisim', '_multisigma', '_SBN_v1', '_SBN_v3', '_SBNNuSyst')


def genie_knob_display_label(knob_name):
    kn = str(knob_name)
    for prefix in _GENIE_KNOB_STRIP_PREFIXES:
        if kn.startswith(prefix):
            kn = kn[len(prefix):]
            break
    for tok in _GENIE_KNOB_STRIP_TOKENS:
        kn = kn.replace(tok, '')
    kn = kn.strip('_')
    while '__' in kn:
        kn = kn.replace('__', '_')
    return kn.replace('_', ' ')


def discover_modes(work_dir):
    modes = []
    merged = Path(work_dir) / 'merged'
    if merged.is_dir():
        for mode in GENIE_GROUP_ORDER:
            if (merged / mode / f'genie_syst_{mode}.npz').is_file():
                modes.append(mode)
    return modes


_MODE_SYST_CACHE = {}


def load_mode_syst(work_dir, mode):
    key = (str(Path(work_dir).resolve()), mode)
    if key in _MODE_SYST_CACHE:
        return _MODE_SYST_CACHE[key]
    npz_path = Path(work_dir) / 'merged' / mode / f'genie_syst_{mode}.npz'
    if not npz_path.is_file():
        _MODE_SYST_CACHE[key] = None
        return None
    syst = np.load(npz_path, allow_pickle=True)['syst'].item()
    _MODE_SYST_CACHE[key] = syst
    return syst


def load_genie_pickle(work_dir):
    pkl = Path(work_dir) / 'syst_disk' / 'GENIE' / 'cov_mat_dict.pkl'
    if not pkl.is_file():
        return None
    with open(pkl, 'rb') as fh:
        return pickle.load(fh)


def mode_genie_pack_from_syst(syst_dict, var_name, mode):
    """Build {rate_parts, xsec_parts, rate_total, xsec_total} for one variable."""
    rate_parts, xsec_parts = {}, {}
    want = set(GENIE_GROUP_KNOBS.get(mode, []))
    for kn, by_var in (syst_dict or {}).items():
        if want and kn not in want:
            continue
        cell = by_var.get(var_name) if isinstance(by_var, dict) else None
        if not isinstance(cell, dict):
            continue
        if isinstance(cell.get('rate'), dict) and cell['rate'].get('cov_frac') is not None:
            rate_parts[kn] = np.asarray(cell['rate']['cov_frac'], dtype=np.float64)
        if isinstance(cell.get('xsec'), dict) and cell['xsec'].get('cov_frac') is not None:
            xsec_parts[kn] = np.asarray(cell['xsec']['cov_frac'], dtype=np.float64)
    return {
        'rate_parts': rate_parts,
        'xsec_parts': xsec_parts,
        'rate_total': _sum_cov_frac_matrices(rate_parts.values()),
        'xsec_total': _sum_cov_frac_matrices(xsec_parts.values()),
    }


def mode_genie_pack_from_pickle(genie_blob, var_name, mode):
    row = genie_blob.get(var_name) if isinstance(genie_blob, dict) else None
    if not isinstance(row, dict):
        return None
    rate_parts, xsec_parts = {}, {}
    for kn in GENIE_GROUP_KNOBS.get(mode, []):
        rk = f'{kn}_rate'
        if rk in row:
            rate_parts[kn] = np.asarray(row[rk], dtype=np.float64)
        if kn in row:
            xsec_parts[kn] = np.asarray(row[kn], dtype=np.float64)
    if not rate_parts and not xsec_parts:
        return None
    return {
        'rate_parts': rate_parts,
        'xsec_parts': xsec_parts,
        'rate_total': _sum_cov_frac_matrices(rate_parts.values()),
        'xsec_total': _sum_cov_frac_matrices(xsec_parts.values()),
    }


def get_matrix_bundle(syst_dict, knob, var_name, kind):
    cell = syst_dict.get(knob, {}).get(var_name, {})
    return cell.get(kind) if isinstance(cell, dict) else None


def _sort_keys_by_score(keys, scores):
    keys = list(keys)
    if not scores:
        return sorted(keys)
    return sorted(keys, key=lambda k: scores.get(k, 0.0), reverse=True)


def _integrated_knob_scores(parts_dict):
    return {kn: integrated_rate_frac_variance(m) for kn, m in (parts_dict or {}).items()}


_STEP_COLOR_POOL = list(plt.get_cmap('tab10').colors) + list(plt.get_cmap('Set2').colors)


def _uncertainty_step_colors(n):
    if n <= 0:
        return []
    out = []
    i = 0
    while len(out) < n:
        out.append(_STEP_COLOR_POOL[i % len(_STEP_COLOR_POOL)])
        i += 1
    return out


def style_uncertainty_axis(ax, var_config, pct_series_list, legend_ncol=3, legend_fontsize=10):
    flat = [np.asarray(s, dtype=np.float64).ravel() for s in pct_series_list if s is not None and len(s)]
    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    xlab = var_config.var_labels[1] if getattr(var_config, 'var_labels', None) else var_config.var_save_name
    ax.set_xlabel(xlab, fontsize=18)
    ax.set_ylabel('Uncertainty [%]', fontsize=18)
    ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
    ax.minorticks_on()
    if getattr(var_config, 'var_save_name', None) == 'integrated':
        ax.set_xticks([])
        ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    fig = ax.get_figure()
    handles, labels = ax.get_legend_handles_labels()
    if labels and len(labels) > LEGEND_OUTSIDE_THRESHOLD:
        fig.subplots_adjust(**BREAKDOWN_SUBPLOT)
        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, BREAKDOWN_SUBPLOT['bottom'] * 0.42),
                   ncol=legend_ncol, fontsize=legend_fontsize, frameon=True)
    elif labels:
        fig.subplots_adjust(**BREAKDOWN_SUBPLOT_ONPLOT)
        ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.18),
                  ncol=min(legend_ncol, len(labels)), fontsize=legend_fontsize, frameon=True)


def plot_knob_breakdown(ax, var_config, parts_dict, total_cov_frac, title='', label_fn=None, knob_sort_scores=None):
    bc = var_config.bin_centers
    bins = var_config.bins
    label_fn = label_fn or genie_knob_display_label
    pct_list = []
    if parts_dict:
        if knob_sort_scores is None and var_config.var_save_name == 'integrated':
            knob_sort_scores = _integrated_knob_scores(parts_dict)
        keys = _sort_keys_by_score(parts_dict.keys(), knob_sort_scores)
        for kn, color in zip(keys, _uncertainty_step_colors(len(keys))):
            w = frac_weights_for_plot(parts_dict[kn], var_config)
            pct_list.append(w)
            ax.hist(bc, bins=bins, weights=w, histtype='step', linewidth=2, color=color, label=label_fn(kn))
    if total_cov_frac is not None:
        wtot = frac_weights_for_plot(total_cov_frac, var_config)
        pct_list.append(wtot)
        ax.hist(bc, bins=bins, weights=wtot, histtype='step', linewidth=2.5, color='k', label='Total')
    ax.set_title(title, fontsize=16)
    style_uncertainty_axis(ax, var_config, pct_list, legend_ncol=4)


def save_figure(fig, out_path, *, dpi=None, save_pdf=None):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    dpi = SUMMARY_SAVE_DPI if dpi is None else dpi
    if save_pdf is None:
        save_pdf = SAVE_SUMMARY_PDF
    fig.savefig(out_path, dpi=dpi, bbox_inches='tight')
    if save_pdf:
        fig.savefig(out_path.with_suffix('.pdf'), bbox_inches='tight')
    if SHOW_FIGS:
        plt.show()
    plt.close(fig)


def _draw_matrix_panel(ax, mat, mkey, subtitle):
    if mat is None or mat.size == 0:
        ax.axis('off')
        ax.text(0.5, 0.5, 'n/a', ha='center', va='center', transform=ax.transAxes)
        return
    mat = np.asarray(mat, dtype=np.float64)
    nb = mat.shape[0]
    extent = [0, nb, 0, nb]
    if mkey == 'corr':
        im = ax.imshow(mat, origin='lower', extent=extent, vmin=-1, vmax=1, cmap='bwr')
    else:
        vmax = np.nanmax(np.abs(mat)) or 1.0
        im = ax.imshow(mat, origin='lower', extent=extent, vmin=-vmax, vmax=vmax, cmap='viridis')
    ax.set_title(subtitle, fontsize=10)
    fig = ax.get_figure()
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


def save_knob_matrix_pdf(mode, knob, syst, var_list, out_pdf):
    out_pdf = Path(out_pdf)
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(out_pdf) as pdf:
        for vc in var_list:
            if vc.var_save_name == 'integrated' and not PLOT_INTEGRATED_MATRICES:
                continue
            bundle_r = get_matrix_bundle(syst, knob, vc.var_save_name, 'rate')
            bundle_x = get_matrix_bundle(syst, knob, vc.var_save_name, 'xsec')
            if bundle_r is None and bundle_x is None:
                continue
            fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=MATRIX_SAVE_DPI)
            title = f'{mode} — {genie_knob_display_label(knob)} — {vc.var_save_name}'
            panels = [
                (bundle_r, 'cov_frac', 'Rate frac. cov.'),
                (bundle_r, 'corr', 'Rate corr.'),
                (bundle_x, 'cov_frac', 'Xsec frac. cov.'),
                (bundle_x, 'corr', 'Xsec corr.'),
            ]
            for ax, (bnd, mkey, subtitle) in zip(np.ravel(axes), panels):
                mat = None if not isinstance(bnd, dict) else bnd.get(mkey)
                nb = len(vc.bins) - 1
                if mat is not None and np.asarray(mat).shape != (nb, nb):
                    mat = None
                _draw_matrix_panel(ax, mat, mkey, subtitle)
            fig.suptitle(title, fontsize=13)
            fig.tight_layout()
            pdf.savefig(fig, dpi=MATRIX_SAVE_DPI)
            plt.close(fig)

In [ ]:
modes = discover_modes(WORK_DIR)
if MODES_TO_PLOT:
    modes = [m for m in MODES_TO_PLOT if m in modes]
genie_pickle = load_genie_pickle(WORK_DIR)

print('Modes with merged NPZ:', modes)
print('Aggregate pickle:', 'yes' if genie_pickle else 'not yet')
for m in modes:
    syst = load_mode_syst(WORK_DIR, m)
    n_kn = len(syst or {})
    reg = len(GENIE_GROUP_KNOBS.get(m, []))
    print(f'  {m}: {n_kn} knobs in NPZ ({reg} registered)')

## Per-knob covariance / correlation matrices

One **multi-page PDF per knob** (`matrices/<MODE>/<knob>__matrices.pdf`): each page is a 2×2 panel (rate/xsec fractional cov + corr) for one variable. Much faster than one PNG/PDF per `(knob, variable)`.

In [ ]:
import time
matrix_manifest = []
t0 = time.time()

for mode in modes:
    syst = load_mode_syst(WORK_DIR, mode)
    if not syst or not PLOT_KNOB_MATRIX_PDFS:
        continue
    knobs = [k for k in GENIE_GROUP_KNOBS.get(mode, []) if k in syst]
    if not knobs:
        knobs = sorted(syst.keys())
    for kn in knobs:
        safe_kn = kn.replace('/', '_')
        out_pdf = OUT_DIR / 'matrices' / mode / f'{safe_kn}__matrices.pdf'
        save_knob_matrix_pdf(mode, kn, syst, var_configs, out_pdf)
        matrix_manifest.append(str(out_pdf))

print(f'Matrix PDFs: {len(matrix_manifest)} in {time.time()-t0:.1f}s')
if matrix_manifest:
    print('example:', matrix_manifest[0])

## Per-mode uncertainty breakdown (rate & xsec)

Per-knob fractional uncertainty curves plus **Total** = sqrt(diag(Σ_k C_k)) for each variable.

In [ ]:
import time
summary_manifest = []
t0 = time.time()

for mode in modes:
    syst = load_mode_syst(WORK_DIR, mode)
    for vc in var_configs:
        pack = None
        if syst:
            pack = mode_genie_pack_from_syst(syst, vc.var_save_name, mode)
        elif genie_pickle:
            pack = mode_genie_pack_from_pickle(genie_pickle, vc.var_save_name, mode)
        if not pack or (not pack['rate_parts'] and not pack['xsec_parts']):
            continue
        rate_scores = _integrated_knob_scores(pack['rate_parts'])
        xsec_scores = _integrated_knob_scores(pack['xsec_parts'])

        if pack['rate_parts']:
            fig, ax = plt.subplots(figsize=BREAKDOWN_FIGSIZE, dpi=BREAKDOWN_FIG_DPI)
            plot_knob_breakdown(
                ax, vc, pack['rate_parts'], pack['rate_total'],
                title=f'{mode} — rate (per knob)',
                knob_sort_scores=rate_scores,
            )
            out = OUT_DIR / 'summary' / mode / f'syst_break_rate__{vc.var_save_name}.png'
            if SAVE_FIGS:
                save_figure(fig, out)
            summary_manifest.append(str(out))

        if pack['xsec_parts']:
            fig, ax = plt.subplots(figsize=BREAKDOWN_FIGSIZE, dpi=BREAKDOWN_FIG_DPI)
            plot_knob_breakdown(
                ax, vc, pack['xsec_parts'], pack['xsec_total'],
                title=f'{mode} — xsec (per knob)',
                knob_sort_scores=xsec_scores,
            )
            out = OUT_DIR / 'summary' / mode / f'syst_break_xsec__{vc.var_save_name}.png'
            if SAVE_FIGS:
                save_figure(fig, out)
            summary_manifest.append(str(out))

print(f'Summary figures: {len(summary_manifest)} in {time.time()-t0:.1f}s')
if summary_manifest:
    print('example:', summary_manifest[0])

In [ ]:
# Integrated uncertainty table (%), per mode × variable
rows = []
for mode in modes:
    syst = load_mode_syst(WORK_DIR, mode)
    for vc in var_configs:
        pack = mode_genie_pack_from_syst(syst, vc.var_save_name, mode) if syst else None
        if not pack:
            continue
        rt = pack['rate_total']
        xt = pack['xsec_total']
        rate_int = 100.0 * np.sqrt(integrated_rate_frac_variance(rt)) if rt is not None else np.nan
        xsec_int = 100.0 * np.sqrt(integrated_rate_frac_variance(xt)) if xt is not None else np.nan
        rows.append((mode, vc.var_save_name, rate_int, xsec_int))

print(f'{"mode":8s} {"variable":16s} {"rate[%]":>10s} {"xsec[%]":>10s}')
for mode, vsn, r, x in rows:
    print(f'{mode:8s} {vsn:16s} {r:10.3f} {x:10.3f}')